# Pelajaran 11 - Protokol Konteks Model (MCP)

**Protokol Konteks Model (MCP)** adalah standar terbuka yang memungkinkan agen untuk secara dinamis menemukan dan menggunakan alat, sumber daya, dan sumber data saat runtime. Alih-alih memasukkan alat secara statis ke dalam agen, MCP memungkinkan agen untuk terhubung ke server eksternal yang memaparkan kemampuan sesuai permintaan.

Dalam pelajaran ini, Anda akan mempelajari:
- Apa itu MCP dan mengapa hal itu penting untuk sistem agen
- Bagaimana arsitektur klien-server MCP bekerja
- Bagaimana membangun agen yang menggunakan penemuan alat bergaya MCP


## Pengaturan

**Persyaratan:**
- Proyek Azure AI Foundry dengan model yang sudah dideploy
- Jalankan `az login` untuk otentikasi


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Apa itu Model Context Protocol (MCP)?

MCP mendefinisikan cara standar bagi agen AI untuk menemukan dan berinteraksi dengan alat dan sumber data eksternal:

- **MCP Server**: Mengekspos alat, sumber daya, dan prompt melalui protokol standar
- **MCP Client**: Runtime agen yang terhubung ke server dan menemukan kemampuan yang tersedia
- **Penemuan Dinamis**: Agen tidak perlu memiliki alat yang dikodekan secara statis — mereka menemukan apa yang tersedia saat runtime

Ini sangat berguna untuk membangun sistem agen yang dapat diperluas di mana kemampuan baru dapat ditambahkan tanpa memodifikasi kode agen.


## Bagaimana MCP Bekerja

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. Agen (klien MCP) terhubung ke server MCP
2. Server merespons dengan daftar alat yang tersedia dan skema mereka
3. Agen kemudian dapat memanggil alat apa pun yang ditemukan selama penalarannya
4. Hasil mengalir kembali melalui protokol yang sama


## Menyimulasikan Penemuan Alat MCP

Karena server MCP yang sebenarnya memerlukan proses server yang berjalan, kami akan mendemonstrasikan pola ini menggunakan fungsi `@tool` yang mensimulasikan apa yang akan disediakan oleh layanan akomodasi yang terhubung ke MCP.

Di lingkungan produksi, alat-alat ini akan ditemukan secara dinamis dari server MCP, bukan didefinisikan secara lokal.


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## Membangun Agen dengan Alat bergaya MCP


In [ ]:
agent = await provider.create_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## MCP in Production

Dalam lingkungan produksi, MCP memungkinkan pola yang kuat:

- **Penemuan alat dinamis**: Agen terhubung ke server MCP dan menemukan alat pada runtime
- **Arsitektur terpisah**: Penyedia alat dapat memperbarui secara independen dari agen
- **Berbagi lintas organisasi**: Tim dapat mengekspos kemampuan melalui server MCP yang dapat digunakan oleh agen mana pun
- **Dukungan Microsoft Agent Framework**: MAF memiliki dukungan klien MCP bawaan melalui integrasi `mcp`

Untuk menggunakan server MCP nyata dengan MAF, Anda akan tersambung melalui `hosted_mcp_tool()` atau integrasi klien MCP.

**Pelajari lebih lanjut:**
- [Spesifikasi MCP](https://modelcontextprotocol.io/)
- [Dukungan MCP Microsoft Agent Framework](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## Ringkasan

Dalam pelajaran ini, Anda mempelajari:
- **MCP** adalah standar terbuka untuk penemuan alat dinamis antara agen dan penyedia alat
- **arsitektur klien-server** memungkinkan agen menemukan kemampuan saat runtime
- MCP memungkinkan **sistem agen yang dapat diperluas dan terpisah** di mana alat dapat ditambahkan tanpa perubahan kode
- Microsoft Agent Framework menyediakan **dukungan MCP bawaan** untuk penggunaan produksi


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Penafian:
Dokumen ini telah diterjemahkan menggunakan layanan terjemahan AI Co-op Translator (https://github.com/Azure/co-op-translator). Meskipun kami berupaya untuk akurat, harap diperhatikan bahwa terjemahan otomatis mungkin mengandung kesalahan atau ketidakakuratan. Dokumen asli dalam bahasa aslinya harus dianggap sebagai sumber yang otoritatif. Untuk informasi yang bersifat kritis, disarankan menggunakan terjemahan profesional oleh penerjemah manusia. Kami tidak bertanggung jawab atas kesalahpahaman atau salah tafsir yang timbul dari penggunaan terjemahan ini.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
